In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("processed/tomato_daily.csv")

In [3]:
df["date"] = pd.to_datetime(df["date"])

In [4]:
full_dates = pd.date_range(
    df["date"].min(),
    df["date"].max(),
    freq="D"
)

df = (
    df.set_index("date")
      .reindex(full_dates)
      .rename_axis("date")
      .reset_index()
)

In [5]:
df["was_imputed"] = df["price"].isna()

In [6]:
missing = df["price"].isna()

In [7]:
groups = (
    missing.ne(missing.shift())
    .cumsum()
)

In [8]:
gap_lengths = (
    missing.groupby(groups)
    .transform("sum")
)

In [9]:
short_gap = missing & (gap_lengths <= 3)

In [10]:
df.loc[short_gap, "price"] = (
    df["price"]
    .interpolate(method="linear")
)

In [11]:
print("Total missing before:", missing.sum())

Total missing before: 280


In [12]:
print(
    "Short-gap values interpolated:",
    short_gap.sum()
)

Short-gap values interpolated: 225


In [13]:
print(
    "Remaining missing values:",
    df["price"].isna().sum()
)

Remaining missing values: 55


In [14]:
df.to_csv(
    "processed/tomato_gbn_model_ready.csv",
    index=False
)

In [15]:
print("\nSaved:")
print("processed/tomato_gbn_model_ready.csv")


Saved:
processed/tomato_gbn_model_ready.csv
